# Day 3: Deploying for Research

**LLMs for Social Science** | Oxford Spring School

## The Course at a Glance

| Day | Theme | What You Learn |
|-----|-------|----------------|
| Day 1 ✓ | Foundations | How models represent meaning, process text, and generate language |
| Day 2 ✓ | From Models to Tools | Post-training (RLHF, DPO), prompting as experimental design, model evaluation |
| **→ Day 3** | **Deploying for Research** | **Validation, fine-tuning (BERT and LoRA), APIs and working at scale** |
| Day 4 | Social Science Applications | Information extraction, RAG, LLMs as simulated agents |
| Day 5 | Agentic Workflows | Tool use, autonomous research assistants, capstone project |

Each day builds on the previous one. Yesterday you classified political tweets with multiple prompting strategies and discovered that prompt wording alone can shift results by 5--15%. Today you learn **how to validate those results rigorously, and when prompting is not enough, how to fine-tune.**

This matters because:

- When you **extract information or build RAG systems** (Day 4), you need structured, validated outputs as inputs to the next stage of your pipeline.
- When you **simulate human respondents** (Day 4), knowing the ceiling of model-human agreement tells you whether simulated data is trustworthy.
- When you **build agentic workflows** (Day 5), fine-tuned models with reliable structured outputs become the backbone of autonomous pipelines.

## Today's outline

- **Section 1:** Validation: Can You Trust Your Results? (~35 min)
- *Break (~15 min)*
- **Section 2:** The Decision Framework and Working at Scale (~20 min)
- **Section 3:** Fine-Tuning for Classification (~65 min)
- **Closing:** The Full Comparison (~10 min)


## Setup

In [ ]:
#@title Install dependencies
!pip install -q transformers accelerate datasets peft trl bitsandbytes
!pip install -q torch pandas scikit-learn tqdm

import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import classification_report, cohen_kappa_score, confusion_matrix, accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")


In [ ]:
#@title Load dataset and Day 2 results

import os

# --- Load the full Women's March dataset ---
DATA_PATH = 'https://raw.githubusercontent.com/antndlcrx/oss_2024/main/data/WM_tweets_groundtruth.csv'
wm_data = pd.read_csv(DATA_PATH)

wm_data['stance_cat'] = wm_data['stance'].map({1: 'support', 0: 'oppose'})
wm_data['sentiment_cat'] = wm_data['sentiment'].map({1.0: 'positive', 0.0: 'negative'})
wm_data['text_cleaned'] = wm_data['text'].str.replace(r'http\S+|www.\S+', '', case=False, regex=True).str.strip()

# --- Create train/val/test splits ---
from sklearn.model_selection import train_test_split

# Use a larger sample for fine-tuning (Day 2 only needed 250 for prompting)
sample_data = wm_data.sample(5000, random_state=42)
train_data, temp_data = train_test_split(sample_data, test_size=0.2, random_state=42, stratify=sample_data['stance_cat'])
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=temp_data['stance_cat'])

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")
print(f"Train stance distribution:\n{train_data['stance_cat'].value_counts()}")

# --- Load Day 2 results (with fallback) ---
DAY2_PATH = 'day2_classification_results.csv'

if os.path.exists(DAY2_PATH):
    day2_results = pd.read_csv(DAY2_PATH)
    print(f"\nLoaded Day 2 results: {len(day2_results)} rows")
    print(f"Columns: {day2_results.columns.tolist()}")
    HAS_DAY2 = True
else:
    print(f"\nDay 2 CSV not found at '{DAY2_PATH}'. Using pre-computed results.")
    print("(This is normal if your runtime restarted since yesterday.)")

    # Pre-computed fallback: approximate results from a Qwen2.5-3B-Instruct run
    # These numbers are representative but will not match any individual student's exact run
    val_subset = wm_data.sample(n=250, random_state=42).reset_index(drop=True)

    np.random.seed(123)
    true_labels = val_subset['stance_cat'].values
    # Simulate ~72% accuracy with a bias toward predicting 'support'
    pred_labels = true_labels.copy()
    flip_idx = np.random.choice(len(pred_labels), size=int(0.28 * len(pred_labels)), replace=False)
    for idx in flip_idx:
        pred_labels[idx] = 'support' if pred_labels[idx] == 'oppose' else 'oppose'

    day2_results = pd.DataFrame({
        'text_cleaned': val_subset['text_cleaned'],
        'stance_cat': true_labels,
        'model_prediction': pred_labels,
        'human_label': [None] * len(val_subset)
    })
    # Simulate 10 human labels (with ~85% human-gold agreement)
    for i in range(10):
        if np.random.random() < 0.85:
            day2_results.loc[i, 'human_label'] = day2_results.loc[i, 'stance_cat']
        else:
            day2_results.loc[i, 'human_label'] = (
                'support' if day2_results.loc[i, 'stance_cat'] == 'oppose' else 'oppose'
            )
    HAS_DAY2 = True

print(f"\nDay 2 results preview:")
print(day2_results.head(3))


---

# Section 1: Validation -- Can You Trust Your Results?

You classified 250 tweets yesterday. The model gave you numbers. But numbers without validation are just numbers.

Social scientists know this already: when you use human coders, you compute inter-coder reliability. When you build a survey instrument, you test for construct validity. LLM-based classification is no different. The model is just another coder, and it needs the same scrutiny.


### Exercise 1a: Accuracy is not enough

Accuracy tells you what fraction of predictions are correct. But with imbalanced classes, accuracy can be misleading. A model that always predicts the majority class gets "high" accuracy while being completely useless.

**Cohen's kappa (κ)** corrects for this by measuring agreement *beyond what you would expect by chance*:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is the observed agreement (accuracy) and $p_e$ is the expected agreement by chance.

- κ = 1: perfect agreement
- κ = 0: no better than chance
- κ < 0: worse than chance (systematic disagreement)

A common interpretation scale: κ > 0.8 is "almost perfect," 0.6--0.8 is "substantial," 0.4--0.6 is "moderate."

Your task: compute both accuracy and κ for the Day 2 model predictions against the gold-standard labels.


In [ ]:
### Exercise 1a: Accuracy vs Cohen's kappa

y_true = day2_results['stance_cat']
y_pred = day2_results['model_prediction']

# Compute accuracy
accuracy = None  # YOUR CODE HERE (hint: accuracy_score from sklearn)

# Compute Cohen's kappa
kappa = None  # YOUR CODE HERE (hint: cohen_kappa_score from sklearn)

print(f"Accuracy: {accuracy:.3f}")
print(f"Cohen's kappa: {kappa:.3f}")
print()
print("Full classification report:")
print(classification_report(y_true, y_pred, digits=3))


In [ ]:
#@title Solution 1a

y_true = day2_results['stance_cat']
y_pred = day2_results['model_prediction']

accuracy = accuracy_score(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"Cohen's kappa: {kappa:.3f}")
print()
print("Full classification report:")
print(classification_report(y_true, y_pred, digits=3))
print()

# Demonstrate why accuracy alone is misleading
class_dist = y_true.value_counts(normalize=True)
majority_class = class_dist.index[0]
majority_pct = class_dist.iloc[0]
print(f"Class distribution: {y_true.value_counts().to_dict()}")
print(f"A model that always predicts '{majority_class}' would get {majority_pct:.1%} accuracy.")
print(f"Its kappa would be 0.0 (no better than chance).")
print()
print("-> Kappa tells you how much better your model is than a naive baseline.")
print("   For research purposes, kappa is almost always the metric you should report.")


### Exercise 1b: Error analysis

Aggregate metrics tell you *how much* the model errs. Error analysis tells you *where* and *why*. This is qualitative coding of quantitative errors, a skill social scientists already have.

Your task: look at the misclassified tweets. Can you find patterns?


In [ ]:
### Exercise 1b: Error analysis

# Identify misclassified tweets
misclassified = day2_results[day2_results['stance_cat'] != day2_results['model_prediction']].copy()
print(f"Misclassified: {len(misclassified)} out of {len(day2_results)} ({100*len(misclassified)/len(day2_results):.1f}%)")
print()

# Look at the confusion matrix: which direction are errors going?
cm = confusion_matrix(y_true, y_pred, labels=['support', 'oppose'])
print("Confusion matrix:")
print(f"              Predicted")
print(f"              support  oppose")
print(f"True support   {cm[0,0]:5d}   {cm[0,1]:5d}")
print(f"True oppose    {cm[1,0]:5d}   {cm[1,1]:5d}")
print()

# YOUR TASK: Examine the misclassified tweets below. What patterns do you see?
# Consider: tweet length, sarcasm, mixed signals, ambiguity.
# Write your observations in the comments.

print("Sample misclassified tweets:")
print("=" * 60)
for _, row in misclassified.head(10).iterrows():
    print(f"  True: {row['stance_cat']} | Predicted: {row['model_prediction']}")
    print(f"  Text: {row['text_cleaned'][:140]}")
    print()

# What patterns do you see?
# YOUR OBSERVATIONS:
#
#
#


In [ ]:
#@title Solution 1b: Common error patterns

print("Common error patterns in LLM classification:")
print()
print("1. SARCASM / IRONY")
print("   A tweet like 'Oh great, another march that will surely fix everything'")
print("   opposes the march but uses positive words. The model picks up on tone,")
print("   not intent.")
print()
print("2. MIXED SIGNALS")
print("   'I respect their right to march but disagree with their message'")
print("   contains both supportive and opposing language. The model must weigh")
print("   competing signals.")
print()
print("3. SHORT / AMBIGUOUS TWEETS")
print("   Very short tweets provide little context. '#WomensMarch' alone could")
print("   be support (sharing) or oppose (mocking).")
print()
print("4. INDIRECT STANCE")
print("   Some tweets express stance through reporting or quoting rather than")
print("   direct statement. The model may classify the reported stance instead")
print("   of the author's stance.")
print()

# Quantitative: are errors biased toward one class?
support_to_oppose = len(misclassified[
    (misclassified['stance_cat'] == 'support') & (misclassified['model_prediction'] == 'oppose')
])
oppose_to_support = len(misclassified[
    (misclassified['stance_cat'] == 'oppose') & (misclassified['model_prediction'] == 'support')
])
print(f"Error direction:")
print(f"  Support misclassified as oppose: {support_to_oppose}")
print(f"  Oppose misclassified as support: {oppose_to_support}")
print()
if oppose_to_support > support_to_oppose:
    print("-> The model has a bias toward predicting 'support'. This is common:")
    print("   LLMs tend to favor the majority class or the more 'positive' label.")
elif support_to_oppose > oppose_to_support:
    print("-> The model has a bias toward predicting 'oppose'.")
else:
    print("-> Errors are roughly balanced between directions.")


### Exercise 1c: How good are YOU?

Here is the most important exercise in this section. You labeled 10 tweets yourself in Day 2. Let's see how well you agree with the gold standard.

This sets a ceiling: if human coders disagree on some tweets, expecting the model to get them right is unreasonable. The gold standard is itself a human judgment.


In [ ]:
### Exercise 1c: Human-human agreement

# Get the tweets that have both human labels and gold labels
human_labeled = day2_results[day2_results['human_label'].notna()].copy()
print(f"Tweets with your manual labels: {len(human_labeled)}")

if len(human_labeled) > 0:
    # Compute agreement between you and the gold standard
    human_kappa = cohen_kappa_score(
        human_labeled['stance_cat'],
        human_labeled['human_label']
    )
    human_accuracy = accuracy_score(
        human_labeled['stance_cat'],
        human_labeled['human_label']
    )

    # Compute agreement between the model and the gold standard (same subset)
    model_kappa_subset = cohen_kappa_score(
        human_labeled['stance_cat'],
        human_labeled['model_prediction']
    )

    print(f"\nYour agreement with gold standard:")
    print(f"  Accuracy: {human_accuracy:.2f}")
    print(f"  Kappa:    {human_kappa:.3f}")
    print()
    print(f"Model agreement with gold standard (same {len(human_labeled)} tweets):")
    print(f"  Kappa:    {model_kappa_subset:.3f}")
    print()

    if human_kappa > model_kappa_subset:
        print("-> You outperformed the model on these tweets. This is typical:")
        print("   humans are better at handling sarcasm, context, and ambiguity.")
    else:
        print("-> The model matched or outperformed you. This can happen with")
        print("   clear-cut tweets where the model's consistency is an advantage.")
    print()
    print("-> The key insight: if your kappa with the gold standard is, say, 0.75,")
    print("   then expecting the model to achieve kappa > 0.75 is unreasonable.")
    print("   That is your inter-coder reliability ceiling.")
else:
    print("No human labels found. If you did not complete the labeling exercise in")
    print("Day 2, you can do it now: look at the first 10 tweets in day2_results")
    print("and fill in the 'human_label' column with 'support' or 'oppose'.")


### Systematic biases in LLM annotation

Beyond random errors, LLMs exhibit *systematic* biases that are distinct from human coder biases:

**Positional bias:** When given options ("support or oppose"), models tend to favor the option listed first. Your Day 2 prompt sensitivity exercise may have shown this: switching "support or oppose" to "oppose or support" can change results.

**Verbosity / sycophancy bias:** Models tend to classify text as the more "positive" or "agreeable" option. In stance detection, this means over-predicting support or agreement.

**Majority-class gravitational pull:** Even with balanced prompts, models often drift toward the class that appears more frequently in their training data. For politically charged topics, this can introduce ideological bias.

**Cultural bias:** Models trained primarily on English-language, Western data may misinterpret cultural expressions, slang, or rhetorical conventions from other contexts.

For social science research, these biases are not just technical annoyances: they are threats to measurement validity. They must be documented and, where possible, corrected.

**Section takeaway:** Treat the LLM as another coder, not as ground truth. Compute inter-annotator agreement (kappa, not just accuracy). Analyze errors qualitatively. Know your ceiling by measuring human-human agreement. Document systematic biases. This validation workflow applies whether you use prompting, fine-tuning, or any other approach.


---

*Take a 15-minute break. When we come back: when to fine-tune, and how.*

---


# Section 2: The Decision Framework and Working at Scale

## When to prompt vs. RAG vs. fine-tune

You now have concrete evidence of how well prompting works for your task, and where it fails. The question is: what do you do next?

**Prompt engineering** (what you did in Day 2) is the right starting point for any task. It requires no training data, runs immediately, and lets you iterate quickly. Stick with prompting when:
- Zero-shot or few-shot performance meets your accuracy threshold
- You need flexibility (changing categories, new tasks)
- Your dataset is small (hundreds, not thousands)
- Reproducibility across prompt variants is acceptable for your purposes

**Retrieval-Augmented Generation (RAG)** is the right choice when the model needs knowledge it does not have: domain-specific documents, your corpus of parliamentary records, legal texts, or any collection too large for the context window. We cover RAG in Day 4.

**Fine-tuning** is the right choice when:
- Prompting hits a ceiling and your validation shows systematic errors
- You need consistent output formatting (always valid JSON, always a specific label set)
- You have enough labeled data (hundreds to thousands of examples)
- Domain-specific language or conventions are not handled well by prompting
- You need a model small enough to run cheaply at scale

The results from Section 1 should help you place your task on this spectrum.


## Working at scale: API patterns (reference)

When you move from experimentation to production, you will typically use a model provider's API rather than running models locally. The code below shows the common patterns. These cells are **reference code** for your own projects: they require API keys that are not available in this session.


In [ ]:
#@title Reference: OpenAI API pattern (requires API key)

# This cell is for reference only. It will not run without an API key.
# To use in your own project: pip install openai, then set OPENAI_API_KEY

EXAMPLE_CODE = '''
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a political text classifier. Respond with only: support or oppose."},
        {"role": "user", "content": f"Does the author support or oppose the Women's March?\n\nTweet: {tweet}"}
    ],
    temperature=0,        # deterministic output
    max_tokens=10,        # we only need one word
)

label = response.choices[0].message.content.strip().lower()
'''

print("OpenAI API pattern:")
print(EXAMPLE_CODE)
print("Cost estimate (gpt-4o-mini): ~$0.15 per 1M input tokens")
print("Classifying 10,000 tweets (~50 tokens each): ~$0.08")


In [ ]:
#@title Reference: Anthropic API pattern (requires API key)

EXAMPLE_CODE = '''
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=10,
    messages=[
        {"role": "user", "content": f"Classify this tweet as supporting or opposing the Women's March. "
                                     f"Answer with one word: support or oppose.\n\nTweet: {tweet}"}
    ],
)

label = message.content[0].text.strip().lower()
'''

print("Anthropic API pattern:")
print(EXAMPLE_CODE)
print("Cost estimate (Claude Sonnet): ~$3.00 per 1M input tokens")
print("Classifying 10,000 tweets (~50 tokens each): ~$1.50")
print("(Claude Haiku is cheaper at ~$0.25 per 1M input tokens)")


In [ ]:
#@title Reference: HuggingFace Inference API pattern (requires API key)

EXAMPLE_CODE = '''
from huggingface_hub import InferenceClient

client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token="hf_..."  # your HuggingFace token
)

response = client.chat_completion(
    messages=[
        {"role": "user", "content": f"Classify this tweet as support or oppose: {tweet}"}
    ],
    max_tokens=10,
)

label = response.choices[0].message.content.strip().lower()
'''

print("HuggingFace Inference API pattern:")
print(EXAMPLE_CODE)
print("Free tier available for many open models.")
print("Pro tier ($9/month) removes rate limits.")
print("Dedicated endpoints available for production use.")


In [ ]:
#@title Reference: Batching with async (for processing thousands of texts)

EXAMPLE_CODE = '''
import asyncio
from openai import AsyncOpenAI

client = AsyncOpenAI()

async def classify_tweet(tweet, semaphore):
    # Classify a single tweet with rate limiting.
    async with semaphore:
        response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Classify as support or oppose. One word only."},
                {"role": "user", "content": f"Tweet: {tweet}"}
            ],
            temperature=0,
            max_tokens=10,
        )
        return response.choices[0].message.content.strip().lower()

async def classify_all(tweets, max_concurrent=20):
    # Process all tweets with controlled concurrency.
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [classify_tweet(t, semaphore) for t in tweets]
    return await asyncio.gather(*tasks)

# Usage:
# results = asyncio.run(classify_all(tweet_list))
'''

print("Async batching pattern (OpenAI example):")
print(EXAMPLE_CODE)
print()
print("Key considerations:")
print("  - Rate limits vary by provider and tier (typically 500-10,000 requests/min)")
print("  - Add retry logic with exponential backoff for rate limit errors")
print("  - Log all responses: API calls are not free to re-run")
print("  - Anthropic also offers a Batch API with 50% cost reduction for non-urgent work")


In [ ]:
#@title Reference: Quick cost estimator

def estimate_cost(n_texts, avg_tokens_per_text=50, output_tokens=10,
                  input_price_per_1M=0.15, output_price_per_1M=0.60):
    """
    Estimate the API cost for a classification run.

    Args:
        n_texts: Number of texts to classify.
        avg_tokens_per_text: Average input tokens per text (tweet ~ 50, paragraph ~ 200).
        output_tokens: Tokens in model response (classification ~ 5-10).
        input_price_per_1M: Price per million input tokens.
        output_price_per_1M: Price per million output tokens.

    Returns:
        Estimated cost in dollars.
    """
    total_input = n_texts * avg_tokens_per_text
    total_output = n_texts * output_tokens
    cost = (total_input / 1_000_000) * input_price_per_1M + \
           (total_output / 1_000_000) * output_price_per_1M
    return cost

# Example estimates
print("Cost estimates for classifying tweets:")
print()
for n in [1_000, 10_000, 100_000]:
    for model, ip, op in [
        ("GPT-4o-mini", 0.15, 0.60),
        ("Claude Haiku", 0.25, 1.25),
        ("Claude Sonnet", 3.00, 15.00),
        ("GPT-4o", 2.50, 10.00),
    ]:
        c = estimate_cost(n, input_price_per_1M=ip, output_price_per_1M=op)
        print(f"  {n:>7,} tweets x {model:<15s} = ${c:>7.2f}")
    print()

print("-> For most classification tasks, GPT-4o-mini or Claude Haiku is the")
print("   cost-effective choice. Use Sonnet/GPT-4o only for hard tasks where")
print("   the quality difference justifies 10-20x the cost.")


---

# Section 3: Fine-Tuning for Classification

You have two tools to learn in this section, each suited to different situations:

**Part A: Encoder fine-tuning (DeBERTa).** When your task is classification and you want maximum accuracy for minimum cost, encoder models like BERT and DeBERTa are the workhorse. They are small, fast, and purpose-built for reading and labeling text.

**Part B: LoRA fine-tuning (generative model).** When you need a generative model (for extraction, summarization, or flexible instruction-following) but want it adapted to your domain, LoRA lets you fine-tune a large model by training only a tiny fraction of its parameters.

Both use the same Women's March dataset, so you can compare them directly against each other and against the prompting results from Day 2.


## Part A: Encoder fine-tuning with DeBERTa

### Why encoders for classification?

In Day 1 you learned about the Transformer architecture. The models you have been using (GPT-2, Qwen) are **decoder-only**: they read text left-to-right and generate one token at a time. This is great for generation, but for classification it is wasteful: the model generates a label token by token when all you need is a single classification decision.

**Encoder models** (BERT, DeBERTa, RoBERTa) read the full text **bidirectionally**: every token attends to every other token simultaneously. This gives them a richer understanding of the text for classification tasks. They are also much smaller (typically 100M--400M parameters vs. 3B+ for generative models), which means faster training and cheaper inference.

DeBERTa-v3-small is our choice: 44M parameters, state-of-the-art for its size, and trains in minutes on a T4.


In [ ]:
#@title Prepare data for DeBERTa fine-tuning

from datasets import Dataset
from transformers import AutoTokenizer

# Convert pandas DataFrames to HuggingFace Datasets
# The Trainer expects a 'label' column with integer labels
label2id = {'oppose': 0, 'support': 1}
id2label = {0: 'oppose', 1: 'support'}

train_data_ft = train_data.copy()
val_data_ft = val_data.copy()
test_data_ft = test_data.copy()

for df in [train_data_ft, val_data_ft, test_data_ft]:
    df['label'] = df['stance_cat'].map(label2id)

train_ds = Dataset.from_pandas(train_data_ft[['text_cleaned', 'label']].rename(columns={'text_cleaned': 'text'}))
val_ds = Dataset.from_pandas(val_data_ft[['text_cleaned', 'label']].rename(columns={'text_cleaned': 'text'}))
test_ds = Dataset.from_pandas(test_data_ft[['text_cleaned', 'label']].rename(columns={'text_cleaned': 'text'}))

# Tokenize
deberta_name = "microsoft/deberta-v3-small"
deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)

def tokenize_fn(examples):
    return deberta_tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

print(f"Tokenized datasets ready.")
print(f"  Train: {len(train_ds)} examples")
print(f"  Val:   {len(val_ds)} examples")
print(f"  Test:  {len(test_ds)} examples")
print(f"  Sample: {train_ds[0]['text'][:80]}...")


### Exercise 3a: Train the classifier

You will configure the training hyperparameters and run fine-tuning. The key choices:

- **Learning rate:** How fast the model updates its weights. Too high and it overshoots; too low and it barely learns. For fine-tuning pre-trained models, 2e-5 to 5e-5 is typical.
- **Epochs:** How many times the model sees the full training set. 2--4 is typical for fine-tuning (more risks overfitting).
- **Batch size:** How many examples the model processes at once. Larger is faster but uses more memory. 16--32 is typical for a T4.

The model trains in about 2 minutes.


In [ ]:
### Exercise 3a: Train DeBERTa classifier

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# Load DeBERTa with a classification head (2 classes: support/oppose)
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_name, num_labels=2, id2label=id2label, label2id=label2id
)

# Metrics function for the Trainer
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

# YOUR CODE: Configure training arguments
training_args = TrainingArguments(
    output_dir="./deberta_stance",
    num_train_epochs=3,           # try 2, 3, or 4
    per_device_train_batch_size=16,  # 16 or 32
    per_device_eval_batch_size=32,
    learning_rate=2e-5,           # try 2e-5 to 5e-5
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",             # disable wandb etc.
    fp16=True,                    # use mixed precision on GPU
)

trainer = Trainer(
    model=deberta_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# Train! (~2 minutes on T4)
trainer.train()


In [ ]:
### Exercise 3b: Evaluate DeBERTa on test set

# Run evaluation on the held-out test set
test_results = trainer.evaluate(test_ds)
print("DeBERTa test set results:")
for k, v in test_results.items():
    if k.startswith('eval_'):
        print(f"  {k.replace('eval_', '')}: {v:.4f}")

# Get per-class breakdown
test_preds = trainer.predict(test_ds)
pred_labels = np.argmax(test_preds.predictions, axis=-1)
true_labels = test_data_ft['label'].values

print("\nPer-class classification report:")
print(classification_report(
    true_labels, pred_labels,
    target_names=['oppose', 'support'], digits=3
))

# Save for later comparison
deberta_accuracy = accuracy_score(true_labels, pred_labels)
deberta_f1_report = classification_report(
    true_labels, pred_labels,
    target_names=['oppose', 'support'], output_dict=True
)
print(f"\nDeBERTa accuracy: {deberta_accuracy:.3f}")


In [ ]:
#@title Free GPU memory for Part B

del deberta_model, trainer
torch.cuda.empty_cache()
import gc
gc.collect()
print("DeBERTa removed from memory.")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## Part B: LoRA fine-tuning of a generative model

### Why LoRA?

DeBERTa is excellent for classification, but it can only classify. If you need a model that can extract information, summarize, or follow flexible instructions tuned to your domain, you need a generative model.

The problem: Qwen2.5-3B has 3 billion parameters. Fine-tuning all of them requires ~24GB of GPU memory and risks catastrophic forgetting (the model loses its general capabilities while learning your task).

**LoRA (Low-Rank Adaptation)** solves this by freezing the entire base model and inserting small trainable matrices (called "adapters") into specific layers. Instead of updating 3 billion parameters, you train ~10 million. This:
- Cuts memory requirements dramatically
- Preserves the model's general capabilities
- Trains in minutes instead of hours
- Produces a small adapter file (tens of MB) that can be layered on top of the base model

**QLoRA** goes further: it loads the base model in 4-bit precision (quantization), reducing memory from ~6GB to ~2.5GB while maintaining quality.


### Exercise 3c: Format the training data

LoRA fine-tuning needs training data formatted as instruction-response pairs. This is the direct connection to Day 2: the "training data" is exactly the kind of prompt-response pairs you were writing and evaluating yesterday.

Your task: write a function that converts each labeled tweet into a conversation format that the model can learn from.


In [ ]:
### Exercise 3c: Format training data as instruction-response pairs

def format_for_sft(row):
    """
    Convert a labeled tweet into a conversation for SFT training.

    The model learns to produce the correct label given the instruction.
    This is exactly what you were doing manually in Day 2 prompting,
    but now the model will learn to do it consistently.

    Args:
        row: A DataFrame row with 'text_cleaned' and 'stance_cat' columns.

    Returns:
        A list of message dicts in the chat format: [{"role": ..., "content": ...}, ...]
    """
    # YOUR CODE HERE
    # Create a list with two dicts:
    #   1. A "user" message containing the classification instruction and the tweet
    #   2. An "assistant" message containing the correct label
    #
    # Hint: use the same instruction style you used in Day 2 prompting.
    # The instruction should be clear and consistent across all examples.

    messages = []  # YOUR CODE HERE

    return messages


# Test on one example
sample_row = train_data.iloc[0]
sample_messages = format_for_sft(sample_row)
print("Formatted example:")
for msg in sample_messages:
    print(f"  [{msg['role']}]: {msg['content'][:120]}")


In [ ]:
#@title Solution 3c

def format_for_sft(row):
    """Convert a labeled tweet into a chat-format training example."""
    messages = [
        {
            "role": "user",
            "content": (
                "Does the author of the following tweet support or oppose the Women's March? "
                "Answer with one word: support or oppose.\n\n"
                f"Tweet: {row['text_cleaned']}"
            )
        },
        {
            "role": "assistant",
            "content": row['stance_cat']
        }
    ]
    return messages


# Test
sample_row = train_data.iloc[0]
sample_messages = format_for_sft(sample_row)
print("Formatted example:")
for msg in sample_messages:
    print(f"  [{msg['role']}]: {msg['content'][:120]}")

# Format all training and validation data
train_conversations = [format_for_sft(row) for _, row in train_data.iterrows()]
val_conversations = [format_for_sft(row) for _, row in val_data.iterrows()]
print(f"\nFormatted {len(train_conversations)} training examples")
print(f"Formatted {len(val_conversations)} validation examples")

print("\n-> Notice: the training data is the same instruction-response format you used")
print("   in Day 2 prompting, but with the CORRECT label as the assistant's response.")
print("   Fine-tuning teaches the model to produce these correct responses consistently.")


In [ ]:
#@title Load Qwen2.5-3B-Instruct with QLoRA (4-bit quantization)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model in 4-bit
model_name = "Qwen/Qwen2.5-3B-Instruct"
lora_tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side='left')
lora_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# Prepare for training
lora_model = prepare_model_for_kbit_training(lora_model)

# Configure LoRA adapters
lora_config = LoraConfig(
    r=16,                    # rank: higher = more capacity, more memory
    lora_alpha=32,           # scaling factor (typically 2x rank)
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # which layers to adapt
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

lora_model = get_peft_model(lora_model, lora_config)

# Report
trainable, total = lora_model.get_nb_trainable_parameters()
print(f"Model loaded in 4-bit with LoRA adapters.")
print(f"  Total parameters:     {total:>12,}")
print(f"  Trainable parameters: {trainable:>12,} ({100*trainable/total:.2f}%)")
print(f"  GPU memory used:      {torch.cuda.memory_allocated() / 1e9:.1f} GB")


### Exercise 3d: Train with LoRA

Training takes approximately 5--10 minutes on a T4 GPU. While it runs, read the explanation below.

**What is happening during training:**

The base model's weights are frozen (the 3 billion parameters do not change). Only the LoRA adapter matrices are being updated. These adapters modify the attention projections (Q, K, V, O matrices) in each transformer layer, gently steering the model's attention patterns toward your task.

The loss curve should decrease steadily. If it plateaus early, the model has learned the task. If it is still decreasing at the end, more epochs might help (but watch for overfitting on the validation loss).


In [ ]:
### Exercise 3d: Train with LoRA (SFTTrainer)

from trl import SFTTrainer, SFTConfig

# Build datasets from the formatted conversations
from datasets import Dataset

def conversations_to_dataset(conversations):
    """Convert list of conversation lists into a HuggingFace Dataset."""
    return Dataset.from_dict({"messages": conversations})

train_sft_ds = conversations_to_dataset(train_conversations)
val_sft_ds = conversations_to_dataset(val_conversations)

# Configure SFT training
sft_config = SFTConfig(
    output_dir="./qwen_lora_stance",
    num_train_epochs=2,
    per_device_train_batch_size=4,       # small batches for memory
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,        # effective batch size = 4 * 4 = 16
    learning_rate=2e-4,                   # LoRA uses higher LR than full fine-tuning
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    max_seq_length=256,                   # tweets are short; saves memory
    report_to="none",
)

trainer = SFTTrainer(
    model=lora_model,
    args=sft_config,
    train_dataset=train_sft_ds,
    eval_dataset=val_sft_ds,
    processing_class=lora_tokenizer,
)

# Train! (~5-10 minutes on T4)
print("Starting LoRA training...")
trainer.train()
print("Training complete.")


In [ ]:
#@title Evaluate LoRA model on test set

# Generate predictions on the test set using the fine-tuned model
lora_model.eval()

def classify_with_lora(texts, model, tokenizer, batch_size=8):
    """Run classification using the LoRA-adapted model."""
    predictions = []

    for i in tqdm(range(0, len(texts), batch_size), desc="LoRA inference"):
        batch_texts = texts[i:i+batch_size]

        # Format as chat messages
        prompts = []
        for text in batch_texts:
            messages = [{"role": "user", "content": (
                "Does the author of the following tweet support or oppose the Women's March? "
                "Answer with one word: support or oppose.\n\n"
                f"Tweet: {text}"
            )}]
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            prompts.append(prompt)

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)

        for j in range(len(batch_texts)):
            prompt_len = inputs['input_ids'][j].ne(tokenizer.pad_token_id).sum()
            resp = tokenizer.decode(outputs[j][prompt_len:], skip_special_tokens=True).lower().strip()
            if "support" in resp:
                predictions.append("support")
            elif "oppose" in resp:
                predictions.append("oppose")
            else:
                predictions.append("unknown")

    return predictions


test_texts = test_data['text_cleaned'].tolist()
lora_preds = classify_with_lora(test_texts, lora_model, lora_tokenizer)

# Evaluate
true_labels_str = test_data['stance_cat'].tolist()
print("LoRA fine-tuned model results:")
print(classification_report(true_labels_str, lora_preds, digits=3))

lora_accuracy = accuracy_score(true_labels_str, lora_preds)
lora_f1_report = classification_report(true_labels_str, lora_preds, output_dict=True)
print(f"LoRA accuracy: {lora_accuracy:.3f}")
print(f"Unknown predictions: {lora_preds.count('unknown')}")


---

# The Full Comparison

### Exercise 3e: Compare all approaches

Let's put everything side by side: the prompting results from Day 2 and the fine-tuning results from today. This is the payoff of two days of work.


In [ ]:
### Exercise 3e: The full comparison

# Collect results from all approaches
print("=" * 75)
print("COMPARISON: All Classification Approaches on the Women's March Dataset")
print("=" * 75)

# Day 2 results (from CSV)
if HAS_DAY2:
    d2_true = day2_results['stance_cat']
    d2_pred = day2_results['model_prediction']
    d2_acc = accuracy_score(d2_true, d2_pred)
    d2_kappa = cohen_kappa_score(d2_true, d2_pred)
    print(f"\nDay 2 prompting (Qwen2.5-3B-Instruct, zero-shot):")
    print(f"  Accuracy: {d2_acc:.3f}  |  Kappa: {d2_kappa:.3f}")
    print(f"  (on {len(d2_true)} validation tweets)")

# DeBERTa results
print(f"\nDeBERTa-v3-small (encoder fine-tuning):")
print(f"  Accuracy: {deberta_accuracy:.3f}")
print(f"  F1 support: {deberta_f1_report['support']['f1-score']:.3f}  |  F1 oppose: {deberta_f1_report['oppose']['f1-score']:.3f}")
print(f"  (on {len(test_data)} test tweets, trained on {len(train_data)} examples)")

# LoRA results
print(f"\nQwen2.5-3B-Instruct + LoRA (generative fine-tuning):")
print(f"  Accuracy: {lora_accuracy:.3f}")
print(f"  F1 support: {lora_f1_report['support']['f1-score']:.3f}  |  F1 oppose: {lora_f1_report['oppose']['f1-score']:.3f}")
print(f"  (on {len(test_data)} test tweets, trained on {len(train_data)} examples)")

print()
print("=" * 75)
print()
print("SUMMARY TABLE")
print()
print(f"{'Approach':<35} {'Accuracy':>10} {'Data needed':>14} {'Training':>12} {'Flexibility':>12}")
print("-" * 85)
if HAS_DAY2:
    print(f"{'Zero-shot prompting':<35} {d2_acc:>10.3f} {'None':>14} {'None':>12} {'High':>12}")
print(f"{'DeBERTa fine-tuned':<35} {deberta_accuracy:>10.3f} {str(len(train_data)):>14} {'~2 min':>12} {'Classify':>12}")
print(f"{'LoRA fine-tuned (generative)':<35} {lora_accuracy:>10.3f} {str(len(train_data)):>14} {'~10 min':>12} {'High':>12}")
print()
print("Observations:")
print("  - Fine-tuning typically outperforms prompting, especially on the minority class.")
print("  - DeBERTa is faster and often more accurate for pure classification tasks.")
print("  - LoRA preserves the model's generative flexibility (extraction, summarization)")
print("    while adapting it to your domain.")
print("  - The choice depends on your task: classification only? Use DeBERTa.")
print("    Need generation + domain adaptation? Use LoRA.")
print("    Quick exploration or no training data? Start with prompting.")


---

# Closing

## What you learned today

1. **Validation is not optional.** Accuracy alone is misleading: use Cohen's kappa, analyze errors qualitatively, and measure human-human agreement to set a realistic ceiling. Treat the LLM as another coder, not as ground truth.

2. **The decision framework matters.** Prompting, RAG, and fine-tuning serve different situations. Start with prompting, validate, and escalate only when you have evidence that a more complex approach is needed.

3. **Encoder fine-tuning (DeBERTa)** is the workhorse for classification: small, fast, accurate, and well-understood. It should be your default for annotation tasks with labeled data.

4. **LoRA fine-tuning** lets you adapt a generative model to your domain without the cost of full fine-tuning. The training data is just prompt-response pairs: the same format you wrote by hand in Day 2.

5. **API patterns** give you the infrastructure to scale any of these approaches to thousands or millions of texts.

| Day | Theme | What You Learn |
|-----|-------|----------------|
| Day 1 ✓ | Foundations | How models represent meaning, process text, and generate language |
| Day 2 ✓ | From Models to Tools | Post-training (RLHF, DPO), prompting as experimental design, model evaluation |
| Day 3 ✓ | Deploying for Research | Validation, fine-tuning (BERT and LoRA), APIs and working at scale |
| **→ Day 4** | **Social Science Applications** | **Information extraction, RAG, LLMs as simulated agents** |
| Day 5 | Agentic Workflows | Tool use, autonomous research assistants, capstone project |

## Bridge to Day 4

You now have a complete pipeline: prompt, validate, and if needed, fine-tune. Day 4 goes beyond classification into three new territories:

- **Information extraction and summarization:** pulling structured data from unstructured text (entities, relationships, arguments) and handling the failure modes (hallucination, omission).
- **Retrieval-Augmented Generation (RAG):** when your corpus is too large for the context window, embeddings (from Day 1) power a retrieval step that feeds relevant documents to the model.
- **LLMs as simulated agents:** the controversial question of whether language models can stand in for human survey respondents, and when that is (and is not) a valid research strategy.

---

Course website: [llmsforsocialscience.net](https://llmsforsocialscience.net)
